In [1]:
import rioxarray
import xarray as xr
import numpy as np
import rasterio
from rasterio.transform import from_bounds
from pathlib import Path

In [2]:
# --- Paths to folders containing your rasters ---
folder1 = Path(r"D:/MyDrive/Stability/RawData/Monthly_Averages/Beaufort_month_year/WeightedMeans_Union")
folder2 = Path(r"D:/MyDrive/Stability/RawData/Monthly_Averages/Chukchi_month_year/WeightedMeans_Union")
output_folder = Path(r"D:/MyDrive/Stability/RawData/Monthly_Averages/MergedOutput_month_year")
output_folder.mkdir(exist_ok=True)

# --- Find common files ---
files1 = {f.name: f for f in folder1.glob("*.tif")}
files2 = {f.name: f for f in folder2.glob("*.tif")}
common_files = set(files1.keys()).intersection(files2.keys())

# --- Function to build union grid ---
def build_union_grid(rasters, res=None):
    bounds = []
    resolutions = []
    for f in rasters:
        da = rioxarray.open_rasterio(f).squeeze()
        bounds.append(da.rio.bounds())
        resolutions.append(da.rio.resolution())
    minx = min(b[0] for b in bounds)
    miny = min(b[1] for b in bounds)
    maxx = max(b[2] for b in bounds)
    maxy = max(b[3] for b in bounds)

    # Pick resolution (finest among all rasters if not specified)
    if res is None:
        resx = min(abs(r[0]) for r in resolutions)
        resy = min(abs(r[1]) for r in resolutions)
    else:
        resx, resy = res

    xs = np.arange(minx, maxx + resx, resx)
    ys = np.arange(maxy, miny - resy, -resy)  # descending
    target = xr.DataArray(
        np.empty((len(ys), len(xs))),
        coords={"y": ys, "x": xs},
        dims=("y", "x"),
    )
    target.rio.write_crs("EPSG:4326", inplace=True)
    return target

# --- Loop through files ---
for f in common_files:
    file1 = files1[f]
    file2 = files2[f]

    # Build union grid for this pair
    target_grid = build_union_grid([file1, file2])

    # Open and reproject/resample to union grid
    da1 = rioxarray.open_rasterio(file1).squeeze().rio.reproject_match(target_grid)
    da2 = rioxarray.open_rasterio(file2).squeeze().rio.reproject_match(target_grid)

    # Convert to numpy arrays
    arr1 = da1.values
    arr2 = da2.values

    # Handle NaN/NoData
    arr1 = np.where(np.isnan(arr1), np.nan, arr1)
    arr2 = np.where(np.isnan(arr2), np.nan, arr2)

    # Merge: prioritize values between 0 and 1
    merged = np.where((arr1 >= 0) & (arr1 <= 1), arr1,
                      np.where((arr2 >= 0) & (arr2 <= 1), arr2, np.nan))

    # Build rasterio metadata
    meta = {
        "driver": "GTiff",
        "height": merged.shape[0],
        "width": merged.shape[1],
        "count": 1,
        "dtype": "float32",
        "crs": target_grid.rio.crs,
        "transform": from_bounds(
            target_grid.x.min(), target_grid.y.min(),
            target_grid.x.max(), target_grid.y.max(),
            merged.shape[1], merged.shape[0]
        ),
        "nodata": 0
    }

    # Write output
    out_file = output_folder / f
    with rasterio.open(out_file, "w", **meta) as dst:
        dst.write(merged.astype(np.float32), 1)
    print(out_file)

print("All rasters merged successfully!")

D:\MyDrive\Stability\RawData\Monthly_Averages\MergedOutput_month_year\WeightedMean_2020_10.tif
D:\MyDrive\Stability\RawData\Monthly_Averages\MergedOutput_month_year\WeightedMean_2019_10.tif
D:\MyDrive\Stability\RawData\Monthly_Averages\MergedOutput_month_year\WeightedMean_2018_05.tif
D:\MyDrive\Stability\RawData\Monthly_Averages\MergedOutput_month_year\WeightedMean_2018_07.tif
D:\MyDrive\Stability\RawData\Monthly_Averages\MergedOutput_month_year\WeightedMean_2017_03.tif
D:\MyDrive\Stability\RawData\Monthly_Averages\MergedOutput_month_year\WeightedMean_2019_02.tif
D:\MyDrive\Stability\RawData\Monthly_Averages\MergedOutput_month_year\WeightedMean_2022_01.tif
D:\MyDrive\Stability\RawData\Monthly_Averages\MergedOutput_month_year\WeightedMean_2018_01.tif
D:\MyDrive\Stability\RawData\Monthly_Averages\MergedOutput_month_year\WeightedMean_2018_10.tif
D:\MyDrive\Stability\RawData\Monthly_Averages\MergedOutput_month_year\WeightedMean_2019_12.tif
D:\MyDrive\Stability\RawData\Monthly_Averages\Merg